<a href="https://colab.research.google.com/github/Ali-Hamza-developer/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit
**Lane: Refresh / Content Opportunity Scoring**

Run top to bottom (Runtime → Run all). Requires `HF_TOKEN` Colab Secret, same as w03-w08.

In [1]:
!pip install -q duckdb scikit-learn

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
print('DuckDB ready, target month =', MONTH)

DuckDB ready, target month = 2026-03


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**This section needs the actual FlyRank research paper text before it can be filled in honestly** — I don't have it in front of me, and per the no-fabrication rule I'm not going to invent findings and then critique them. Paste the paper (or the two sections/findings you want audited) and I'll fill this in properly. The template below is the shape to fill:

**Finding 1:** *[quote or closely paraphrase the paper's claim, in your own words — see the citation/copyright note below]*
- **Where the label comes from:** *[observed outcome in the warehouse data, e.g. `trend_direction` / `future_decline`? A defined rule? A rebuilt product flag? State it plainly.]*
- **Does the validation design carry the claim?** *[What split did the paper use — random, grouped, time-aware? Does that split match the strength of the claim being made? A causal-sounding claim needs more than a same-month random split can support.]*
- **My question, framed constructively:** *[e.g. "Was the reported number computed on a client-held-out test set, or could some of these clients also appear in training?"]*

**Finding 2:** *[same structure]*
- **Where the label comes from:**
- **Does the validation design carry the claim?**
- **My question, framed constructively:**

One note in advance, regardless of which two findings you pick: if the paper reports a single accuracy/AUC number without saying whether the split was grouped by client, that's worth asking about directly — Section 2 below shows concretely how much a naive vs. grouped split can move the number on this exact dataset.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-8 capstone (`w05_model.ipynb`) already used a client-grouped split. To make the "before/after" honest and visible, this section rebuilds the same feature set and label, then trains the **same model architecture** twice: once on a **naive random row split** (the kind of split that would silently let the same client's content appear in both train and test) and once on the **client-grouped split** actually used in the capstone. Both are scored on the same two metrics — AUC and Precision@50 — so the effect of the split choice on this exact dataset is visible directly, not asserted.

In [2]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Rebuild the same honest feature base + label as w05_model.ipynb (ML-08)
base = con.sql(f"""
    WITH window_90d AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_90d,
            SUM(gsc_clicks) AS clicks_90d,
            AVG(gsc_avg_position) AS avg_position_90d
        FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        w.client_hash_id,
        w.content_hash_id,
        LOG(1 + w.impressions_90d) AS log_impressions_90d,
        w.avg_position_90d,
        CASE WHEN w.impressions_90d > 0 THEN w.clicks_90d::DOUBLE / w.impressions_90d ELSE NULL END AS ctr_90d,
        (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) AS days_since_last_update,
        d.word_count,
        d.content_type,
        d.main_intent,
        d.competition_level
    FROM window_90d w
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON w.content_hash_id = d.content_hash_id AND w.client_hash_id = d.client_hash_id
    WHERE w.impressions_90d > 0
      AND (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) >= 0
""").df()

label_window = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_next30
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    WHERE report_date <= DATE '2026-04-30'
    GROUP BY 1, 2
""").df()

modeling_df = base.merge(label_window, on=['client_hash_id', 'content_hash_id'], how='inner')
modeling_df['future_decline'] = (
    modeling_df['impressions_next30'] < 0.7 * (2 ** modeling_df['log_impressions_90d'] - 1)
).astype(int)
modeling_df = modeling_df.drop(columns=['impressions_next30'])

features = ['log_impressions_90d', 'avg_position_90d', 'ctr_90d', 'days_since_last_update', 'word_count',
            'content_type', 'main_intent', 'competition_level']
cat_cols = ['content_type', 'main_intent', 'competition_level']

def precision_at_k(y_true, y_score, k=50):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    order = np.argsort(-y_score)[:k]
    return y_true[order].mean()

def fit_and_score(train_df, test_df):
    tr, te = train_df.copy(), test_df.copy()
    for c in cat_cols:
        tr[c] = tr[c].astype('category')
        te[c] = te[c].astype(pd.CategoricalDtype(categories=tr[c].cat.categories))
    cat_idx = [features.index(c) for c in cat_cols]
    clf = HistGradientBoostingClassifier(categorical_features=cat_idx, random_state=42)
    clf.fit(tr[features], tr['future_decline'])
    scores = clf.predict_proba(te[features])[:, 1]
    auc = roc_auc_score(te['future_decline'], scores)
    p50 = precision_at_k(te['future_decline'], scores, k=50)
    return auc, p50

# BEFORE: naive random row split — ignores that many rows share a client
train_naive, test_naive = train_test_split(modeling_df, test_size=0.25, random_state=42)
auc_naive, p50_naive = fit_and_score(train_naive, test_naive)

# AFTER: honest split — grouped by client_hash_id, same protocol as w05_model.ipynb
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(modeling_df, groups=modeling_df['client_hash_id']))
train_honest, test_honest = modeling_df.iloc[tr_idx], modeling_df.iloc[te_idx]
auc_honest, p50_honest = fit_and_score(train_honest, test_honest)

before_after = pd.DataFrame({
    'split': ['naive random row split (before)', 'grouped-by-client split (after, honest)'],
    'AUC': [round(auc_naive, 3), round(auc_honest, 3)],
    'Precision@50': [round(p50_naive, 3), round(p50_honest, 3)],
    'test_clients_also_in_train': [
        len(set(test_naive['client_hash_id']).intersection(set(train_naive['client_hash_id']))),
        len(set(test_honest['client_hash_id']).intersection(set(train_honest['client_hash_id']))),
    ],
})
print(before_after.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                  split   AUC  Precision@50  test_clients_also_in_train
        naive random row split (before) 0.924          0.82                          46
grouped-by-client split (after, honest) 0.889          0.58                           0


**Fill in after running:** state the actual before/after AUC and Precision@50 numbers from the table above. Did the naive split's `test_clients_also_in_train` count come back non-zero (confirming client overlap) or zero (meaning the random split happened to separate clients cleanly anyway on this particular seed)? If the naive AUC is meaningfully higher than the grouped one, say so plainly — that gap is the honest cost of not accounting for client structure, not a modeling failure.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same three attacks as the Week-5 leakage hunt (`w03_feature_leakage_check.ipynb`), re-run here on the **final feature set** from Section 2 rather than the earlier draft feature vector, and scored on the honest grouped split rather than a random one:

1. A future-window column built directly from the label period.
2. A label-derived column that's a near-copy of the target.
3. A rebuilt product-style priority flag built from current-window signals only (not a future leak, but excluded per the lane guide since it would just teach the model to copy an existing rule).

In [3]:
from sklearn.linear_model import LogisticRegression

# Re-attach the label-window sum ONLY for this attack cell — never used for real training elsewhere
attack_df = modeling_df.merge(label_window, on=['client_hash_id', 'content_hash_id'], how='left')
attack_df['impressions_next30'] = attack_df['impressions_next30'].fillna(0)

# ATTACK 1 — future-window column, built directly from the label period
attack_df['next_30d_impressions_delta'] = (
    attack_df['impressions_next30'] - (2 ** attack_df['log_impressions_90d'] - 1)
)

# ATTACK 2 — label-derived column, near-copy of the target itself
attack_df['decline_flag_leak'] = attack_df['future_decline']

# ATTACK 3 — rebuilt product-style priority flag from CURRENT-window signals only
attack_df['rebuilt_priority_flag'] = (
    (attack_df['avg_position_90d'] > 10).astype(int) + (attack_df['ctr_90d'].fillna(0) < 0.02).astype(int)
)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx2, te_idx2 = next(gss2.split(attack_df, groups=attack_df['client_hash_id']))
train_a, test_a = attack_df.iloc[tr_idx2].copy(), attack_df.iloc[te_idx2].copy()

def fit_auc_simple(train_df, test_df, feats):
    d_train = train_df.dropna(subset=feats + ['future_decline'])
    d_test = test_df.dropna(subset=feats + ['future_decline'])
    clf = LogisticRegression(max_iter=1000)
    clf.fit(d_train[feats], d_train['future_decline'])
    scores = clf.predict_proba(d_test[feats])[:, 1]
    return roc_auc_score(d_test['future_decline'], scores)

features_numeric_honest = ['log_impressions_90d', 'avg_position_90d', 'ctr_90d',
                            'days_since_last_update', 'word_count']
attack_configs = {
    'final feature set (honest, grouped split)': features_numeric_honest,
    '+ future-window leak': features_numeric_honest + ['next_30d_impressions_delta'],
    '+ label-derived leak': features_numeric_honest + ['decline_flag_leak'],
    '+ rebuilt product-flag leak': features_numeric_honest + ['rebuilt_priority_flag'],
}
for name, feats in attack_configs.items():
    auc = fit_auc_simple(train_a, test_a, feats)
    print(f"{name:42s} AUC = {auc:.3f}")

print("\nKeep only the honest AUC on the model card — the rest exist to prove the leaks, not to report.")

final feature set (honest, grouped split)  AUC = 0.837
+ future-window leak                       AUC = 1.000
+ label-derived leak                       AUC = 1.000
+ rebuilt product-flag leak                AUC = 0.836

Keep only the honest AUC on the model card — the rest exist to prove the leaks, not to report.


**Fill in after running:** did the same pattern hold as in Week 5 — future-window and label-derived leaks inflate AUC sharply, while the rebuilt product-flag leak moves it little or not at all? Quote the actual AUC values. If any number surprised you relative to Week 5 (e.g. a smaller or larger jump now that the split is grouped by client instead of random), say so — that's a legitimate finding about how the grouped split interacts with these specific leaks, not a mistake to hide.

**This needs your actual boldest sentence** — from the paper draft, the portfolio copy, or wherever you wrote the most confident claim about what this model does. I don't want to invent a strawman overclaim and then "fix" it, since that wouldn't reflect what you actually wrote. Drop the sentence in and I'll do the rewrite with you.

The pattern to follow, using a claim you've already used elsewhere as a **worked example** of language that already sits on the safe side of this line:

| | Original | Rewritten |
|---|---|---|
| Example (already safe) | "I build ML systems to a metric I define and defend — not just train and hope." | *(no rewrite needed — this is a claim about your process and standards, not a claim about what the model proves)* |
| **Your sentence** | *[paste it here]* | *[fill in together]* |

General moves for the rewrite, worth checking against your sentence once it's in:
- Swap causal-sounding verbs ("predicts," "proves," "drives") for observational ones ("is associated with," "was observed to," "correlates with").
- Attach the number to its actual test conditions instead of stating it as a universal fact — e.g. "AUC of X on a client-held-out test set" rather than "X% accurate."
- Replace "will" with "is intended to support" or "provides decision support for" when describing what the model does for the analyst — matches the careful-words standard used throughout w01-w08.
- If the sentence implies the model works for any client or any content type, check whether that's actually been tested (Section 2 above is direct evidence either way) before letting the sentence generalize that far.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.